# 11.3 Coding Practice: Working with Speech in Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module11/week11_speech_practice.ipynb)

This is a guided, ungraded activity. Follow the steps in order and run each cell. Along the way you will turn text into speech, look at sound as numbers and as pictures, prepare audio for a speech recognizer, turn the speech back into text, and measure the errors.

The activity follows Readings [11.1](https://obscrivn.github.io/mynewbook/module11/speech_signals.html) and [11.2](https://obscrivn.github.io/mynewbook/module11/speech_recognition.html). It runs on a CPU (no GPU needed) and takes about 40 to 50 minutes.

**What you will do**

1. Make speech from text with gTTS, play it, and convert MP3 to WAV.
2. Load the WAV file as numbers and draw its waveform.
3. See the sound as a spectrogram and as a log-Mel spectrogram.
4. Get the audio ready for a model: mono, 16,000 samples per second, and a telephone-quality copy.
5. Transcribe the audio with two speech recognizers.
6. Measure the errors with word error rate (WER), including with telephone audio and background noise.
7. Optional: repeat everything with your own voice.

## Setup

Run the next cell to install the packages this activity needs. In Google Colab this takes about a minute. The second line installs gTTS on its own, without its extra dependencies, so it does not change the versions of packages Colab already has.

In [ ]:
%pip -q install "transformers==5.17.0" "jiwer==4.0.0" "pydub==0.25.1" "audioop-lts; python_version >= '3.13'"
%pip -q install --no-deps "gTTS==2.5.4"

Now import the libraries. `soundfile` reads and writes audio files, `librosa` analyzes audio, `pydub` converts between audio formats, `jiwer` computes word error rate, and `transformers` gives us ready-made speech recognizers.

In [ ]:
import os
import re
import urllib.request

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # keep the output tidy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import librosa.display
import jiwer
import transformers
from transformers import pipeline
from pydub import AudioSegment
from IPython.display import Audio, display

transformers.logging.set_verbosity_error()
print("transformers", transformers.__version__, "| librosa", librosa.__version__, "| jiwer 4.0.0")

## 1. From text to speech (about 5 minutes)

We start with **text-to-speech (TTS)**: the reverse of speech recognition (Reading 11.2, Section 1). The `gTTS` library sends your text to Google Translate's text-to-speech service and saves the spoken result as an MP3 file.

Run the cell below. It saves the sentence as `welcome.mp3`. (In Colab, files are saved in the `/content` folder, which you can see in the file panel on the left.)

If the service cannot be reached, the cell downloads a copy of the same file from the course website instead, so you can always continue.

In [ ]:
# Import Google Text-To-Speech
from gtts import gTTS

# The text that you want to convert to audio
mytext = 'Welcome to the big hotel California'

# Language in which you want to convert
language = 'en'

# Passing the text and language to the engine, and
# saving the converted audio in an mp3 file named welcome
COURSE_COPY = "https://raw.githubusercontent.com/obscrivn/mynewbook/master/module11/data/welcome.mp3"
try:
    myobj = gTTS(text=mytext, lang=language)
    myobj.save("welcome.mp3")
    print("Created welcome.mp3 with gTTS.")
except Exception as error:
    print("gTTS could not reach Google's service:", type(error).__name__)
    print("Downloading the course copy of welcome.mp3 instead.")
    urllib.request.urlretrieve(COURSE_COPY, "welcome.mp3")
    mytext = 'Welcome to the big hotel California'  # the sentence in the course copy

Listen to it. `autoplay=True` starts playing as soon as the cell runs; you can also press play.

In [ ]:
display(Audio('welcome.mp3', autoplay=True))

MP3 is a **compressed** format: it throws away detail that listeners are unlikely to notice, which makes files small. Most audio analysis tools, and the speech models later in this activity, work with **WAV**, which stores every sample as a plain number. The `pydub` library converts between the two (it uses the `ffmpeg` program, which Colab already has).

In [ ]:
from pydub import AudioSegment
sound = AudioSegment.from_mp3("welcome.mp3")
sound.export("welcome.wav", format="wav")

print(f"welcome.mp3: {os.path.getsize('welcome.mp3'):>8,} bytes (compressed)")
print(f"welcome.wav: {os.path.getsize('welcome.wav'):>8,} bytes (every sample stored)")

The WAV file holds the same sound but is several times larger, because nothing is compressed.

**Try it (optional):** change `mytext` to your own sentence and run the three cells above again. You can also try another language, for example `mytext = 'Bienvenidos al gran hotel'` with `language = 'es'`, or a different English accent by changing the gTTS line to `gTTS(text=mytext, lang='en', tld='co.uk')`. The rest of the activity works with whatever sentence you choose, because it uses `mytext` as the correct transcript.

## 2. Sound as numbers (about 5 minutes)

A WAV file is a long list of numbers: one **sample** of air pressure for each tick of the clock (Reading 11.1, Section 4). Load it with `soundfile` and look at the basic facts.

In [ ]:
audio, sr = sf.read("welcome.wav")

print("Sampling rate:     ", f"{sr:,} samples per second")
print("Number of samples: ", f"{len(audio):,}")
print("Duration:          ", round(len(audio) / sr, 2), "seconds")
print("Channels:          ", 1 if audio.ndim == 1 else audio.shape[1], "(mono)" if audio.ndim == 1 else "")
print("First 10 samples:  ", np.round(audio[:10], 4))
print("Lowest and highest:", round(float(audio.min()), 3), round(float(audio.max()), 3))

What to notice:

- gTTS produces **24,000 samples per second**. Multiply the duration by 24,000 and you get the number of samples.
- `soundfile` scales the samples to numbers between -1 and 1. Zero means normal air pressure (silence).
- The first samples are close to zero: the recording starts with a little silence.

Now draw the **waveform**: time on the horizontal axis, amplitude on the vertical axis.

In [ ]:
times = np.arange(len(audio)) / sr

plt.figure(figsize=(12, 3))
plt.plot(times, audio, linewidth=0.6)
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title(f"Waveform: '{mytext}'")
plt.tight_layout()
plt.show()

Taller stretches are louder, mostly vowels. Flat stretches are silence or quiet consonants. You cannot read the words off this picture, but you can see where the speech is.

Zoom in on 30 milliseconds from the loudest part of the recording. Each dot is one sample.

In [ ]:
window = int(0.030 * sr)  # 30 milliseconds, in samples
loudness = librosa.feature.rms(y=audio, frame_length=window, hop_length=window // 2)[0]
start = int(np.argmax(loudness)) * (window // 2)
segment = audio[start:start + window]

plt.figure(figsize=(12, 3))
plt.plot((start + np.arange(len(segment))) / sr * 1000, segment, marker=".", markersize=3, linewidth=0.8)
plt.xlabel("Time (milliseconds)")
plt.ylabel("Amplitude")
plt.title("30 ms close-up of the loudest part")
plt.tight_layout()
plt.show()

f0, voiced, _ = librosa.pyin(audio, fmin=60, fmax=400, sr=sr)
print(f"Typical pitch (fundamental frequency, F0) of this voice: about {np.nanmedian(f0):.0f} Hz")

The close-up shows a pattern that **repeats**: each repeat is one opening and closing of the vocal folds during a vowel. The number of repeats per second is the fundamental frequency (F0), which we hear as pitch. `librosa.pyin` estimates it for the whole recording.

## 3. Seeing sound: spectrograms (about 10 minutes)

A waveform shows *how much* the air moves. A **spectrogram** shows *which frequencies* are present at each moment (Reading 11.1, Section 5):

- horizontal axis: time
- vertical axis: frequency in hertz, low at the bottom
- darkness: how much energy there is at that frequency and time (in this first plot, **darker means more energy**, as in the figures in Reading 11.1)

The next cell defines a small helper that draws a waveform and a spectrogram one above the other, so the times line up. We will reuse it several times.

In [ ]:
def plot_waveform_and_spectrogram(y, sr, title, max_hz=None):
    """Draw the waveform (top) and spectrogram (bottom) of y on the same time axis."""
    win = int(0.006 * sr)            # a short 6 ms window shows formants as broad bands
    hop = int(0.002 * sr)
    spec_db = librosa.amplitude_to_db(
        np.abs(librosa.stft(y, n_fft=512, win_length=win, hop_length=hop)), ref=np.max)

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, layout="constrained",
                             gridspec_kw={"height_ratios": [1, 2]})
    librosa.display.waveshow(y, sr=sr, ax=axes[0], linewidth=0.6)
    axes[0].set(title=f"Waveform: {title}", ylabel="Amplitude", xlabel="")
    img = librosa.display.specshow(spec_db, sr=sr, hop_length=hop, x_axis="time", y_axis="hz",
                                   cmap="gray_r", ax=axes[1])
    axes[1].set(title="Spectrogram (darker = more energy)", xlabel="Time (seconds)", ylabel="Frequency (Hz)")
    if max_hz:
        axes[1].set_ylim(0, max_hz)
    fig.colorbar(img, ax=axes, format="%+2.0f dB", shrink=0.6)
    plt.show()


plot_waveform_and_spectrogram(audio, sr, f"'{mytext}'")

How to read this picture:

- **The top of the frequency axis is 12,000 Hz**, half of the 24,000 Hz sampling rate. A recording cannot contain frequencies above half its sampling rate (the Nyquist frequency, Reading 11.1, Section 4).
- **Vowels** show dark horizontal bands in the lower part of the plot, called **formants**. Their positions change from vowel to vowel (and glide during *wel-*, *ho-*, and *Cali-*); that is how vowels are told apart. The thin vertical lines inside vowels are the individual pulses of the vocal folds.
- **Hissing sounds** such as the *f* in *California* show up as fuzzy energy spread up into the high frequencies, without clear bands.
- **Stop consonants** such as the *b* and *g* in *big* and the *t* in *hotel* show up as short pale gaps followed by a thin vertical burst.
- **There are no gaps between words.** Pale gaps come from stop consonants and pauses, not from word boundaries.

Zoom into the speech range below 5,000 Hz to see the formants more clearly:

In [ ]:
plot_waveform_and_spectrogram(audio, sr, f"'{mytext}' (0 to 5,000 Hz)", max_hz=5000)

### What a speech model actually reads: the log-Mel spectrogram

Speech recognizers do not read this plain spectrogram. They read a **log-Mel spectrogram** (Reading 11.1, Section 5): frames of 25 milliseconds taken every 10 milliseconds, with frequencies grouped on the Mel scale (more detail at low frequencies, like human hearing) and energy on a log scale.

The cell below computes it with 80 Mel bands, the setting used by Whisper. This time the plot uses a color map, which is common in speech tools: **brighter means more energy**.

In [ ]:
frame = int(0.025 * sr)   # 25 ms window
hop = int(0.010 * sr)     # a new frame every 10 ms
mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=frame, hop_length=hop, n_mels=80)
log_mel = librosa.power_to_db(mel, ref=np.max)

print("Log-Mel spectrogram shape:", log_mel.shape, "-> (Mel bands, frames)")
print(f"{len(audio) / sr:.2f} seconds at one frame every 10 ms is about {len(audio) / sr * 100:.0f} frames, "
      "plus one because librosa centers the first frame at time zero")

fig, ax = plt.subplots(figsize=(12, 4))
img = librosa.display.specshow(log_mel, sr=sr, hop_length=hop, x_axis="time", y_axis="mel", cmap="magma", ax=ax)
ax.set(title="Log-Mel spectrogram (80 Mel bands; brighter = more energy)",
       xlabel="Time (seconds)", ylabel="Frequency (Hz, Mel-spaced)")
fig.colorbar(img, ax=ax, format="%+2.0f dB")
plt.show()

This grid of 80 numbers per 10 milliseconds is the "text" a speech recognizer reads. Notice how the Mel axis gives more room to low frequencies: the formant bands take up much more of the picture than in the plain spectrogram.

## 4. Getting audio ready for a model (about 5 minutes)

The speech models in Part 5 were trained on **mono audio at 16,000 samples per second**. Our file is at 24,000. Recordings from phones and computers come at many rates (44,100 and 48,000 are common) and are often stereo, so it is worth having one function that handles any WAV file.

The function below reads a file **without changing it** and returns a new mono copy at 16,000 Hz. The original file stays exactly as it was.

In [ ]:
TARGET_SR = 16000


def load_for_model(path, target_sr=TARGET_SR):
    """Read an audio file without changing it, and return a mono copy at target_sr samples per second."""
    original, original_sr = sf.read(path, always_2d=True)  # shape: (samples, channels)
    print(f"{path}: {original_sr:,} Hz, {original.shape[1]} channel(s), "
          f"{original.shape[0] / original_sr:.2f} seconds, {original.shape[0]:,} samples")
    mono = original.mean(axis=1)                              # average the channels
    if original_sr != target_sr:
        mono = librosa.resample(mono, orig_sr=original_sr, target_sr=target_sr)
    print(f"  model copy: {target_sr:,} Hz, mono, {len(mono):,} samples")
    return mono.astype(np.float32)


audio_16k = load_for_model("welcome.wav")
sf.write("welcome_16k.wav", audio_16k, TARGET_SR)  # saved under a new name; welcome.wav is untouched
display(Audio(audio_16k, rate=TARGET_SR))

The duration is the same, but there are fewer samples: duration × 16,000 instead of duration × 24,000. It sounds the same, because almost all of the speech information is below 8,000 Hz, the highest frequency a 16,000 Hz recording can hold.

### Telephone quality

Telephone audio uses only **8,000 samples per second**, so it keeps frequencies up to about 4,000 Hz (Reading 11.1, Section 6). The function below imitates that by going down to 8,000 samples per second and back up to 16,000. Listen to the result and compare the two spectrograms.

In [ ]:
def telephone(y, sr=TARGET_SR):
    """Imitate telephone audio: keep only frequencies below about 4,000 Hz."""
    narrow = librosa.resample(y, orig_sr=sr, target_sr=8000)
    return librosa.resample(narrow, orig_sr=8000, target_sr=sr).astype(np.float32)


audio_phone = telephone(audio_16k)
display(Audio(audio_phone, rate=TARGET_SR))

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
win, hop = int(0.006 * TARGET_SR), int(0.002 * TARGET_SR)
for ax, (y, label) in zip(axes, [(audio_16k, "16,000 Hz copy"), (audio_phone, "Telephone quality")]):
    spec_db = librosa.amplitude_to_db(
        np.abs(librosa.stft(y, n_fft=512, win_length=win, hop_length=hop)), ref=np.max)
    librosa.display.specshow(spec_db, sr=TARGET_SR, hop_length=hop, x_axis="time", y_axis="hz", cmap="gray_r", ax=ax)
    ax.set(title=label, xlabel="Time (seconds)", ylabel="")
axes[0].set_ylabel("Frequency (Hz)")
plt.tight_layout()
plt.show()

In the telephone version, everything above about 4,000 Hz is gone. Listen for the *f* in *California*: much of its hiss lived up there, which is why *f* and *s* are easy to confuse on the phone.

## 5. From speech back to text (about 10 minutes)

Now we go the other way: **speech recognition**. We load two pretrained recognizers from the Hugging Face Hub (Reading 11.2, Section 4):

- **Whisper** (`openai/whisper-base.en`): an attention **encoder-decoder**. It reads the whole clip and writes text one token at a time, with capital letters and punctuation.
- **wav2vec 2.0** (`facebook/wav2vec2-base-960h`): a self-supervised encoder fine-tuned with **CTC**. It outputs one letter (or a blank) for every short slice of audio, then cleans up.

The first run downloads both models (a few hundred megabytes), which takes a minute or two.

In [ ]:
whisper = pipeline("automatic-speech-recognition", model="openai/whisper-base.en",
                   device=-1, model_kwargs={"disable_mmap": True})
wav2vec = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-base-960h",
                   device=-1, model_kwargs={"disable_mmap": True})


def transcribe(y, sr=TARGET_SR):
    """Transcribe a mono audio array with both models."""
    return {
        "Whisper (encoder-decoder)": whisper({"raw": y.copy(), "sampling_rate": sr})["text"].strip(),
        "wav2vec 2.0 (CTC)": wav2vec({"raw": y.copy(), "sampling_rate": sr})["text"].strip(),
    }


results = transcribe(audio_16k)
print(f"{'Correct text':<27} {mytext}")
for model_name, text in results.items():
    print(f"{model_name:<27} {text}")

Compare the two outputs:

- **Whisper** writes text the way a person would: capital letters, punctuation, and often digits for numbers. Its decoder works like a language model, so it produces well-formed sentences.
- **wav2vec 2.0** writes only **UPPERCASE letters**, with no punctuation. Its CTC output is built from letters chosen slice by slice, so it has no built-in sense of sentence formatting.

In the run saved with this page, Whisper recognized every word, while wav2vec 2.0 wrote *HOTAL* for *hotel*. CTC spells what it hears letter by letter, and nothing inside it knows that *hotal* is not an English word; Whisper's decoder, which works like a language model, would rarely write a non-word. Your run may differ slightly. Part 6 shows how to score both outputs fairly.

## 6. Measuring errors (about 10 minutes)

Word error rate compares a model's output (the **hypothesis**) with the correct text (the **reference**) and counts substitutions (S), deletions (D), and insertions (I) (Reading 11.2, Section 5):

WER = (S + D + I) / number of words in the reference

Here the reference is `mytext`, the sentence we gave gTTS. First, score the outputs exactly as they are.

In [ ]:
reference = mytext

for model_name, text in results.items():
    print(f"{model_name:<27} raw WER = {jiwer.wer(reference, text):.2f}")

The raw scores count differences in capital letters and punctuation as word errors: *Welcome* and *WELCOME* are different strings to a computer. Before scoring, we **normalize** both texts in the same way. The function below lowercases the text and removes punctuation while keeping apostrophes (so *don't* stays one word).

Numbers need their own rule in real projects (*14* versus *fourteen*), and so do contractions (*don't* versus *do not*). Whatever rules you choose, apply them to the reference and the hypothesis alike, and report them.

In [ ]:
def normalize(text):
    """Lowercase, remove punctuation (keep apostrophes), and tidy spaces."""
    text = text.lower()
    text = re.sub(r"[^\w\s']", " ", text)
    return " ".join(text.split())


for model_name, text in results.items():
    score = jiwer.wer(normalize(reference), normalize(text))
    print(f"{model_name:<27} normalized WER = {score:.2f}   ({normalize(text)})")

After normalization, only real word differences remain. `jiwer` can also show the alignment, marking each substitution (S), deletion (D), and insertion (I):

In [ ]:
for model_name, text in results.items():
    alignment = jiwer.process_words(normalize(reference), normalize(text))
    print(model_name)
    print(f"  substitutions={alignment.substitutions}, deletions={alignment.deletions}, "
          f"insertions={alignment.insertions}, correct={alignment.hits}")
    print(jiwer.visualize_alignment(alignment, show_measures=False))

### Telephone quality and background noise

Real recordings are rarely this clean. The next cell adds **white noise** (a steady hiss) so that the speech is 10 decibels louder than the noise, roughly a quiet voice in a noisy café. The random seed is fixed, so everyone gets the same noise. Listen to it, then look at its spectrogram.

In [ ]:
def add_noise(y, snr_db=10, seed=0):
    """Add white noise so the speech is snr_db decibels louder than the noise."""
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal(len(y))
    speech_power = np.mean(y ** 2)
    noise *= np.sqrt(speech_power / (np.mean(noise ** 2) * 10 ** (snr_db / 10)))
    return (y + noise).astype(np.float32)


audio_noisy = add_noise(audio_16k, snr_db=10)
display(Audio(audio_noisy, rate=TARGET_SR))
plot_waveform_and_spectrogram(audio_noisy, TARGET_SR, "with background noise (10 dB)")

The noise fills the whole spectrogram with gray, burying the quiet consonants first; the strong vowel bands still stand out. Now transcribe all three versions with both models and put the results in one table.

In [ ]:
def evaluate(versions, reference):
    """Transcribe each version with both models and score it against the reference."""
    rows = []
    for condition, y in versions.items():
        for model_name, text in transcribe(y).items():
            rows.append({"condition": condition, "model": model_name, "transcript": text,
                         "normalized WER": round(jiwer.wer(normalize(reference), normalize(text)), 2)})
    return pd.DataFrame(rows)


versions = {"clean (16 kHz)": audio_16k, "telephone": audio_phone, "noise 10 dB": audio_noisy}
table = evaluate(versions, reference)
pd.set_option("display.max_colwidth", None)
display(table)

Read the table row by row. In the run saved with this page:

- **Telephone quality** changed nothing for this short, clear, synthetic sentence. Real voices on real phone lines usually fare worse.
- **Noise** changed wav2vec 2.0's output from *HOTAL CALIFORNIA* to *HOTEL TALIFORNIAN*: it fixed one word and broke another. The *k* sound at the start of *California* is a short, quiet burst, exactly the kind of sound noise covers first. The WER stayed at 0.17, so the number alone would hide that a different word went wrong.

In general:

- Look at **which words change**, not only at the WER number. Errors usually hit short, quiet sounds and words first.
- The two models can react differently to the same audio, which is why it is worth testing a model on audio like your own before relying on it.
- One sentence is far too little to compare models fairly. Real evaluations use many recordings, many speakers, and the same normalization rules for every system.

**Try it (optional):** change `snr_db=10` to `snr_db=0` (speech and noise equally loud) in the noise cell and run the cells again. Or go back to Part 1, give gTTS a sentence with a number and a negation, such as `'The number 14 bus does not stop here after 9:30.'`, and see how each model writes the numbers.

## 7. Optional: try it with your own voice

If you can record audio, you can run the same steps on your own speech. Synthetic speech is clean and standard; real voices are not, so expect more interesting results. This part is optional, and nothing is submitted.

**What to say.** About 10 to 20 seconds (stay under 30 seconds). Use a **fictional, everyday sentence** with a number or a negation, for example: *"The number 14 bus does not stop at Maple Street after 9:30 at night."* or *"I can't find the blue umbrella, and the shop closes at 6."*

**Do not record personal information**: no real names, addresses, phone numbers, ID or account numbers, health details, or anything else that could identify you or someone else.

**How to record**

- **Audacity** (free for Windows, macOS, and Linux, <https://www.audacityteam.org/>): press Record, read your sentence, press Stop, then choose **File → Export Audio…**, Format **WAV**, Encoding **Signed 16-bit PCM**. Any sample rate, mono or stereo, is fine.
- **A phone or computer recorder** (Voice Memos, Voice Recorder, QuickTime) usually saves **.m4a**. That is fine too: the cell below converts .m4a or .mp3 to WAV with `pydub`, the same way Part 1 converted the MP3.
- Record in a quiet room, 15 to 30 cm from the microphone. Avoid online converter websites; they upload your voice to a third party.

**Privacy.** Files you upload to Colab are deleted when the session ends. Audio players in this notebook store the sound inside the saved notebook file, so do not share or post a notebook that contains your voice.

**Steps**

1. In the next cell, set `USE_MY_RECORDING = True`, and type **exactly what you said** in `MY_SENTENCE`. This is your reference transcript, so write it before you look at any model output.
2. Run the cell. In Colab it opens an upload button; choose your file. Outside Colab, put the file next to this notebook and set `MY_FILE` to its name.
3. Run the cell after it to repeat Parts 2 to 6 on your recording. Your original file is never changed.

In [ ]:
USE_MY_RECORDING = False   # change to True to use your own recording
MY_SENTENCE = ""           # type exactly what you said
MY_FILE = ""               # outside Colab: the file name, for example "my_recording.wav"

if USE_MY_RECORDING and not MY_FILE:
    try:
        from google.colab import files
        uploaded = files.upload()
        MY_FILE = next(iter(uploaded))
    except ImportError:
        print("Not running in Colab: set MY_FILE to the name of your audio file.")

if USE_MY_RECORDING:
    print("Using:", MY_FILE)

In [ ]:
if USE_MY_RECORDING:
    name, extension = os.path.splitext(MY_FILE)
    if extension.lower() == ".wav":
        my_wav = MY_FILE
    else:
        # Convert .m4a, .mp3, and similar formats to WAV, saved under a new name
        my_wav = name + "_converted.wav"
        AudioSegment.from_file(MY_FILE).export(my_wav, format="wav")
        print(f"Converted {MY_FILE} to {my_wav}")

    my_audio = load_for_model(my_wav)
    display(Audio(my_audio, rate=TARGET_SR))
    plot_waveform_and_spectrogram(my_audio, TARGET_SR, "my recording (16 kHz copy)")

    my_versions = {"clean (16 kHz)": my_audio,
                   "telephone": telephone(my_audio),
                   "noise 10 dB": add_noise(my_audio, snr_db=10)}
    display(evaluate(my_versions, MY_SENTENCE))
else:
    print("Skipped: set USE_MY_RECORDING = True in the cell above to use your own voice.")

## Wrap-up

You followed the full path of speech data:

**text → speech (TTS) → MP3 → WAV → numbers → waveform → spectrogram → log-Mel spectrogram → 16 kHz mono model input → text (ASR) → word error rate**

Along the way you saw that a recording's sampling rate limits which frequencies it can hold, that spectrograms show vowels, hissing sounds, and stops, that two recognizers can write the same words in very different forms, that normalization decides what counts as an error, and that telephone audio and noise change what a model hears.

These are the same steps used in real speech projects, such as transcribing interviews, meetings, or customer calls. The usual next steps are to use more recordings and more speakers, choose normalization rules carefully, and look at *which* errors happen, not just how many. The Week 11 discussion picks up from here.